# Day 49 — Practice solutions

Run in order. The setup mirrors the question notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(19)
torch.manual_seed(7)
print('PyTorch', torch.__version__)

## Setup

In [ ]:
torch.manual_seed(3)
X = torch.randn(20, 2)
y = torch.randint(0, 2, (20, 1)).float()
model = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
criterion = nn.BCEWithLogitsLoss()

## 1. Shapes

(20, 2) → (20, 8) → (20, 8) → (20, 1).

In [ ]:
shapes = [tuple(X.shape), tuple(model[0](X).shape), tuple(model[1](model[0](X)).shape), tuple(model(X).shape)]
print(shapes)

## 2. Parameter count

First layer: 8×2 + 8 = 24. Last layer: 1×8 + 1 = 9. Total = 33.

In [ ]:
parameter_count = sum(p.numel() for p in model.parameters())
print(parameter_count)
assert parameter_count == 33

## 3. Logits and probabilities

The last Linear layer is unbounded. Sigmoid maps its logits to (0,1).

In [ ]:
logits = model(X)
probs = torch.sigmoid(logits)
print(logits[:3,0], probs[:3,0])

## 4. Correct binary loss

Pass raw logits to BCEWithLogitsLoss and float targets of identical shape.

In [ ]:
assert logits.shape == y.shape
loss = criterion(logits, y)
print(loss.item())

## 5. Backward pass

Gradients have the same shape as their parameters.

In [ ]:
model.zero_grad()
criterion(model(X), y).backward()
gradient_shape = tuple(model[0].weight.grad.shape)
print(gradient_shape)
assert gradient_shape == (8,2)

## 6. One SGD update

The optimizer uses the gradients already computed in question 5.

In [ ]:
before = model[0].weight.detach().clone()
optimizer = torch.optim.SGD(model.parameters(), lr=.1)
optimizer.step()
changed = not torch.equal(before, model[0].weight.detach())
print(changed)
assert changed

## 7. Why zero_grad?

Autograd accumulates parameter gradients. Without reset, the second backward call adds to the first.

In [ ]:
model.zero_grad()
criterion(model(X), y).backward()
first = model[0].weight.grad.detach().clone()
criterion(model(X), y).backward()
second = model[0].weight.grad.detach().clone()
print(torch.allclose(second, 2*first))
assert torch.allclose(second, 2*first)
model.zero_grad()

## 8. Validation mode

Use eval() for inference behavior and no_grad() to avoid recording a graph.

In [ ]:
model.eval()
with torch.no_grad():
    validation_probs = torch.sigmoid(model(X))
print(validation_probs.shape)

## 9. Accuracy

Compare thresholded predictions with float labels, then average.

In [ ]:
pred = (validation_probs >= .5).float()
accuracy = (pred == y).float().mean().item()
print(accuracy)

## 10. Avoid leakage

Standardize validation data with training statistics. Validation never contributes to mean or standard deviation.

In [ ]:
train_np = np.array([[1.,2.],[3.,6.],[5.,4.]])
val_np = np.array([[10.,20.]])
train_mean = train_np.mean(axis=0)
train_std = train_np.std(axis=0)
val_scaled = (val_np-train_mean)/train_std
print(train_mean, train_std, val_scaled)

## 11. Save in memory

Rebuild the same architecture and load its state dictionary.

In [ ]:
import io
buffer = io.BytesIO()
torch.save(model.state_dict(), buffer)
buffer.seek(0)
reloaded = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
reloaded.load_state_dict(torch.load(buffer, map_location='cpu', weights_only=True))
reloaded.eval()
with torch.no_grad():
    assert torch.allclose(model(X), reloaded(X))
print('same logits')

## 12. Experiment

For 2 → 16 → 1, the first weight is (16,2). Total parameters: 16×2+16+16+1=65.

In [ ]:
new_model = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 1))
new_count = sum(p.numel() for p in new_model.parameters())
print(tuple(new_model[0].weight.shape), new_count)
assert new_count == 65